In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    DT, SIM_TIME, VEHICLE_LENGTH, MIN_GAP,
    HDV_ACCEL_MIN, HDV_ACCEL_MAX, HDV_SPEED_MAX,
    AV_ACCEL_MIN, AV_ACCEL_MAX, AV_SPEED_MAX,
)

FIG_WIDTH = 12
FIG_HEIGHT = 3

print(f"DT = {DT} s")
print(f"Simulation time = {SIM_TIME} s")
print(f"Vehicle length = {VEHICLE_LENGTH} m")
print(f"Minimum gap = {MIN_GAP} m")

In [ ]:
from src.dataloader import load_data

df = load_data()

print("Initial traffic configuration:")
display(df)
print(f"\nNumber of vehicles: {len(df)}")
print("Vehicle types:")
print(df["type"].value_counts())

In [ ]:
from src.dynamics import calculate_gap, HDVDynamics

# Use the same gap calculation as the simulation.
df["gap_m"] = np.nan

for i in range(1, len(df)):
    df.loc[i, "gap_m"] = calculate_gap(
        df.loc[i - 1, "position_m"],
        df.loc[i, "position_m"]
    )

print(df[["vehicle_id", "position_m", "speed_mps", "type", "gap_m"]].to_string(index=False))


In [ ]:
valid_gaps = df["gap_m"].dropna()

if (valid_gaps <= 0).any():
    print("WARNING: At least one vehicle has a non-positive initial gap.")
else:
    print("All initial vehicle gaps are positive.")

print(f"Minimum initial gap: {valid_gaps.min():.2f} m")
print(f"Mean initial gap:    {valid_gaps.mean():.2f} m")
print(f"Maximum initial gap: {valid_gaps.max():.2f} m")


In [ ]:
# Initial HDV accelerations using the same model as src.dynamics.
df["initial_accel_mps2"] = np.nan

for i in range(1, len(df)):
    if df.loc[i, "type"] == "HDV":
        df.loc[i, "initial_accel_mps2"] = HDVDynamics.calculate_acceleration(
            df.loc[i, "speed_mps"],
            df.loc[i - 1, "speed_mps"],
            df.loc[i, "gap_m"]
        )

display(df[[
    "vehicle_id", "type", "position_m", "speed_mps",
    "gap_m", "initial_accel_mps2"
]])

hdv_acc = df["initial_accel_mps2"].dropna()
if not hdv_acc.empty:
    print(f"Minimum HDV acceleration: {hdv_acc.min():.2f} m/s²")
    print(f"Maximum HDV acceleration: {hdv_acc.max():.2f} m/s²")


In [ ]:
fig, ax = plt.subplots(figsize=(FIG_WIDTH, FIG_HEIGHT))

VEHICLE_HEIGHT = 0.2
ROAD_Y = 0.0 # Y-coordinate of the road
ROAD_HEIGHT = VEHICLE_HEIGHT + 0.2

# Road
ax.add_patch(
    plt.Rectangle(
        (df["position_m"].min() - 10, -ROAD_HEIGHT / 2),         # bottom-left corner of the road
        df["position_m"].max() - df["position_m"].min() + 20,    # length of the road in x 
        ROAD_HEIGHT,                                             # width of the road in y
        facecolor="lightgray",
        edgecolor="lightgray",
        linewidth=1.0
    )
)

# Vehicles
for _, row in df.iterrows():
    x = row["position_m"]
    facecolor = "tab:red" if row["type"] == "AV" else "tab:blue"
    ax.add_patch(
        plt.Rectangle(
            (x - VEHICLE_LENGTH, ROAD_Y - VEHICLE_HEIGHT / 2), # bottom-left corner of the vehicle bounding box
            VEHICLE_LENGTH,
            VEHICLE_HEIGHT,
            facecolor=facecolor,
            edgecolor="black",
            linewidth=1.0
        )
    )

    ax.text(
        x - VEHICLE_LENGTH / 2,
        VEHICLE_HEIGHT / 2 + 0.18,
        f"{int(row['vehicle_id'])}\n"
        f"{row['speed_mps']:.1f} m/s",
        ha="center",
        va="bottom",
        fontsize=8
    )

# Direction
ax.text(
    0.98, 0.08,
    "Direction of travel →",
    transform=ax.transAxes,
    ha="right",
    va="bottom",
    fontsize=9
)

# Axes
ax.set_xlim(
    df["position_m"].min() - 10,
    df["position_m"].max() + 10
)

ax.set_ylim(-1.8, 1.8)

ax.set_xlabel("Position along road [m]")
ax.set_yticks([])


ax.set_title("Initial Traffic Configuration")

ax.grid(axis="x", alpha=0.2)

# Legend
from matplotlib.patches import Patch

legend_elements = [
    Patch(facecolor="tab:blue", edgecolor="black", label="HDV"),
    Patch(facecolor="tab:red", edgecolor="black", label="AV")
]

ax.legend(
    handles=legend_elements,
    loc="lower left"
)

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(FIG_WIDTH, 4.5))

for vehicle_type, marker, label in [
    ("HDV", "o", "HDV"),
    ("AV", "s", "AV"),
]:
    subset = df[df["type"] == vehicle_type]
    ax.scatter(
        subset["position_m"], subset["speed_mps"],
        marker=marker, s=80, label=label
    )

    for _, row in subset.iterrows():
        ax.annotate(
            int(row["vehicle_id"]),
            (row["position_m"], row["speed_mps"]),
            xytext=(5, 5), textcoords="offset points", fontsize=8
        )

ax.set_xlabel("Initial position [m]")
ax.set_ylabel("Initial speed [m/s]")
ax.set_title("Initial Vehicle Positions and Speeds")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
from src.simulation import simulate

# Run the updated mixed-traffic simulation.
results = simulate(df)

time = results["time"]
positions = results["positions"]
speeds = results["speeds"]
accelerations = results["accelerations"]
gaps = results["gaps"]
vehicle_types = results["vehicle_types"]

pedestrian_state = results["pedestrian_state"]
pedestrian_position = results["pedestrian_position"]

print(f"Simulation steps: {len(time)}")
print(f"Simulation duration: {time[-1]:.1f} s")
print(f"Vehicles: {positions.shape[1]}")
print("Vehicle types:", vehicle_types)

print(f"\nPedestrian position: {pedestrian_position:.1f} m")
print("Pedestrian states:")
print(
    f"  Warning: 18–20 s"
    f"\n  Active:  20–30 s"
    f"\n  Cleared: after 30 s"
)

In [ ]:
# -------------------------------------------------
# Pedestrian state check
# -------------------------------------------------

print("===== PEDESTRIAN STATE CHECK =====")

for state in ["none", "warning", "active", "cleared"]:
    indices = np.where(pedestrian_state == state)[0]

    if len(indices) > 0:
        print(
            f"{state:>8}: "
            f"{time[indices[0]]:.1f} s "
            f"to "
            f"{time[indices[-1]]:.1f} s"
        )

In [ ]:
# -------------------------------------------------
# HDV pedestrian response diagnostic
# -------------------------------------------------

from src.config import (
    PEDESTRIAN_HDV_WARNING_BRAKE,
    PEDESTRIAN_HDV_ACTIVE_BRAKE,
)

pedestrian_acceleration = results["pedestrian_acceleration"]

print("===== HDV PEDESTRIAN RESPONSE =====")

hdv_indices = np.where(vehicle_types == "HDV")[0]

for i in hdv_indices:

    vehicle_id = int(df.iloc[i]["vehicle_id"])

    warning_mask = (
        (pedestrian_state == "warning")
        & np.isfinite(pedestrian_acceleration[:, i])
    )

    active_mask = (
        (pedestrian_state == "active")
        & np.isfinite(pedestrian_acceleration[:, i])
    )

    print(f"\nVehicle {vehicle_id}:")

    if np.any(warning_mask):
        k = np.where(warning_mask)[0][0]

        print(
            f"  Warning response starts: "
            f"t = {time[k]:.1f} s, "
            f"position = {positions[k, i]:.2f} m, "
            f"speed = {speeds[k, i]:.2f} m/s, "
            f"pedestrian acceleration = "
            f"{pedestrian_acceleration[k, i]:.2f} m/s²"
        )
    else:
        print("  No pedestrian warning response.")

    if np.any(active_mask):
        k = np.where(active_mask)[0][0]

        print(
            f"  Active response starts: "
            f"t = {time[k]:.1f} s, "
            f"position = {positions[k, i]:.2f} m, "
            f"speed = {speeds[k, i]:.2f} m/s, "
            f"pedestrian acceleration = "
            f"{pedestrian_acceleration[k, i]:.2f} m/s²"
        )
    else:
        print("  No pedestrian active response.")

In [ ]:
# -------------------------------------------------
# Pedestrian event: HDV accelerations
# -------------------------------------------------

fig, ax = plt.subplots(figsize=(FIG_WIDTH, 5))

for i in hdv_indices:

    vehicle_id = int(df.iloc[i]["vehicle_id"])

    ax.plot(
        time,
        accelerations[:, i],
        label=f"Vehicle {vehicle_id} (HDV)"
    )

ax.axvspan(
    18.0, 20.0,
    alpha=0.15,
    label="Pedestrian warning"
)

ax.axvspan(
    20.0, 30.0,
    alpha=0.15,
    label="Pedestrian active"
)

ax.axhline(
    PEDESTRIAN_HDV_WARNING_BRAKE,
    linestyle="--",
    linewidth=1.0,
    label="Warning brake"
)

ax.axhline(
    PEDESTRIAN_HDV_ACTIVE_BRAKE,
    linestyle="--",
    linewidth=1.0,
    label="Active brake"
)

ax.set_xlim(15, 35)
ax.set_xlabel("Time [s]")
ax.set_ylabel("Acceleration [m/s²]")
ax.set_title("HDV Response to Pedestrian")
ax.grid(alpha=0.3)
ax.legend(ncol=2)

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(FIG_WIDTH, 6))

for i in range(positions.shape[1]):
    vehicle_id = int(df.iloc[i]["vehicle_id"])
    vehicle_type = vehicle_types[i]

    ax.plot(
        time, positions[:, i],
        label=f"Vehicle {vehicle_id} ({vehicle_type})",
        linewidth=2 if vehicle_type == "AV" else 1.2,
    )

ax.set_xlabel("Time [s]")
ax.set_ylabel("Position [m]")
ax.set_title("Vehicle Position Trajectories")
ax.grid(True, alpha=0.3)
ax.legend(ncol=2, fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(FIG_WIDTH, 6))

for i in range(speeds.shape[1]):
    vehicle_id = int(df.iloc[i]["vehicle_id"])
    vehicle_type = vehicle_types[i]

    ax.plot(
        time, speeds[:, i],
        label=f"Vehicle {vehicle_id} ({vehicle_type})",
        linewidth=2 if vehicle_type == "AV" else 1.2,
    )

ax.set_xlabel("Time [s]")
ax.set_ylabel("Speed [m/s]")
ax.set_title("Vehicle Speed Trajectories")
ax.grid(True, alpha=0.3)
ax.legend(ncol=2, fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Safety and physical checks
ordering_ok = np.all(positions[:, :-1] > positions[:, 1:], axis=1)

print("Vehicle ordering maintained:", ordering_ok.all())

if not ordering_ok.all():
    first_bad = np.where(~ordering_ok)[0][0]
    print(f"First ordering violation: t = {time[first_bad]:.1f} s")

minimum_gap = np.nanmin(gaps)
idx = np.unravel_index(np.nanargmin(gaps), gaps.shape)

print(f"Minimum simulated gap: {minimum_gap:.3f} m")
print(f"Minimum gap occurs at t = {time[idx[0]]:.1f} s, vehicle index = {idx[1]}")
print(f"Required minimum gap: {MIN_GAP:.3f} m")

print(f"Maximum speed: {speeds.max():.3f} m/s")
print(f"Minimum speed: {speeds.min():.3f} m/s")
print(f"Number of zero-speed states: {np.sum(speeds <= 0.0)}")

In [ ]:
fig, ax = plt.subplots(figsize=(FIG_WIDTH, 6))

for i in range(1, gaps.shape[1]):
    vehicle_id = int(df.iloc[i]["vehicle_id"])
    vehicle_type = vehicle_types[i]

    ax.plot(
        time, gaps[:, i],
        label=f"Vehicle {vehicle_id} ({vehicle_type})",
        linewidth=2 if vehicle_type == "AV" else 1.2,
    )

ax.axhline(
    MIN_GAP, linestyle="--", linewidth=1.5,
    label=f"Minimum gap = {MIN_GAP} m"
)

ax.set_xlabel("Time [s]")
ax.set_ylabel("Gap [m]")
ax.set_title("Vehicle Gaps")
ax.grid(True, alpha=0.3)
ax.legend(ncol=2, fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(FIG_WIDTH, 6))

for i in range(accelerations.shape[1]):
    vehicle_id = int(df.iloc[i]["vehicle_id"])
    vehicle_type = vehicle_types[i]

    ax.plot(
        time, accelerations[:, i],
        label=f"Vehicle {vehicle_id} ({vehicle_type})",
        linewidth=2 if vehicle_type == "AV" else 1.2,
    )

ax.axhline(0, linewidth=0.8)
ax.set_xlabel("Time [s]")
ax.set_ylabel("Acceleration [m/s²]")
ax.set_title("Vehicle Accelerations")
ax.grid(True, alpha=0.3)
ax.legend(ncol=2, fontsize=8)
plt.tight_layout()
plt.show()


In [ ]:
summary = pd.DataFrame({
    "vehicle_id": df["vehicle_id"].astype(int),
    "type": vehicle_types,
    "max_speed_mps": np.max(speeds, axis=0),
    "min_speed_mps": np.min(speeds, axis=0),
    "max_accel_mps2": np.max(accelerations, axis=0),
    "min_accel_mps2": np.min(accelerations, axis=0),
    "min_gap_m": [
        np.nanmin(gaps[:, i]) if i > 0 else np.nan
        for i in range(len(vehicle_types))
    ],
})

display(summary)

In [ ]:
# -------------------------------------------------
# AV time-headway diagnostic
# -------------------------------------------------

from src.config import AV_HEADWAY_TARGET, AV_EPSILON

time = results["time"]
gaps = results["gaps"]
speeds = results["speeds"]
vehicle_types = results["vehicle_types"]

av_indices = np.where(vehicle_types == "AV")[0]

print("===== AV TIME-HEADWAY DIAGNOSTIC =====")

for i in av_indices:

    valid = gaps[:, i] > 0

    headway = np.full_like(time, np.nan, dtype=float)

    headway[valid] = (
        gaps[valid, i]
        / np.maximum(speeds[valid, i], AV_EPSILON)
    )

    k = np.nanargmin(headway)

    vehicle_id = int(df.iloc[i]["vehicle_id"])

    print(
        f"Vehicle {vehicle_id}: "
        f"min = {headway[k]:.3f} s "
        f"at t = {time[k]:.1f} s, "
        f"gap = {gaps[k, i]:.3f} m, "
        f"speed = {speeds[k, i]:.3f} m/s, "
        f"mean = {np.nanmean(headway):.3f} s, "
        f"final = {headway[-1]:.3f} s"
    )

In [ ]:
# -------------------------------------------------
# AV time-headway plot
# -------------------------------------------------

fig, ax = plt.subplots(figsize=(FIG_WIDTH, FIG_HEIGHT))

for i in av_indices:

    valid = gaps[:, i] > 0

    headway = np.full_like(time, np.nan, dtype=float)

    headway[valid] = (
        gaps[valid, i]
        / np.maximum(speeds[valid, i], AV_EPSILON)
    )

    vehicle_id = int(df.iloc[i]["vehicle_id"])

    ax.plot(
        time,
        headway,
        label=f"Vehicle {vehicle_id} (AV)"
    )

ax.axhline(
    AV_HEADWAY_TARGET,
    linestyle="--",
    label=f"Target = {AV_HEADWAY_TARGET:.1f} s"
)

ax.set_xlabel("Time [s]")
ax.set_ylabel("Time headway [s]")
ax.set_title("AV Time Headway")
ax.grid(alpha=0.2)
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# -------------------------------------------------
# Longitudinal jerk
# -------------------------------------------------

# Jerk is the rate of change of acceleration
jerk = np.diff(accelerations, axis=0) / DT

# Jerk has one fewer sample than acceleration
jerk_time = time[1:]

fig, ax = plt.subplots(figsize=(FIG_WIDTH, FIG_HEIGHT))

for i in range(len(vehicle_types)):

    vehicle_id = int(df.iloc[i]["vehicle_id"])

    if vehicle_types[i] == "AV":
        ax.plot(
            jerk_time,
            jerk[:, i],
            label=f"Vehicle {vehicle_id} (AV)",
            linewidth=1.5
        )
    else:
        ax.plot(
            jerk_time,
            jerk[:, i],
            alpha=0.5,
            linewidth=1.0
        )

# Comfort reference limits
ax.axhline(
    2.0,
    linestyle="--",
    linewidth=1.0,
    label="+2 m/s³ reference"
)

ax.axhline(
    -2.0,
    linestyle="--",
    linewidth=1.0,
    label="-2 m/s³ reference"
)

ax.set_xlabel("Time [s]")
ax.set_ylabel("Longitudinal jerk [m/s³]")
ax.set_title("Longitudinal Jerk")

ax.grid(alpha=0.2)
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
print("===== LONGITUDINAL JERK SUMMARY =====")

for i in range(len(vehicle_types)):

    vehicle_id = int(df.iloc[i]["vehicle_id"])

    abs_jerk = np.abs(jerk[:, i])

    print(
        f"Vehicle {vehicle_id} ({vehicle_types[i]}): "
        f"max |jerk| = {np.max(abs_jerk):.2f} m/s³, "
        f"mean |jerk| = {np.mean(abs_jerk):.2f} m/s³, "
        f"above 2 m/s³ = "
        f"{100 * np.mean(abs_jerk > 2):.1f}%"
    )